# Gold Layer — Training, Forecasting & Anomaly Detection

Trains MV AR-LSTM, evaluates on held-out test set, retrains on full data, produces 30-day forecast with Gaussian noise, and flags anomalies (95th percentile).

**Note:** Metrics have not yet been measured on the Indian dataset.

| Column | Type | Description |
|--------|------|-------------|
| `DATE` | `datetime64` | Date |
| `CITY` | `str` | State or union territory name |
| `TEMPERATURE_C` | `float64` | Actual or forecasted |
| `PRECIPITATION_MM` | `float64` | Actual or forecasted |
| `WIND_SPEED_KMH` | `float64` | Actual or forecasted |
| `SOURCE` | `str` | `historical` or `forecast` |
| `ANOMALY_TEMPERATURE` | `bool` | Exceeds 95th percentile |
| `ANOMALY_PRECIPITATION` | `bool` | Exceeds 95th percentile |
| `ANOMALY_WIND` | `bool` | Exceeds 95th percentile |

In [ ]:
df = spark.read.table("silver_weather_india.weather.processed_weather")
df = df.toPandas()
df.columns = [c.upper() for c in df.columns]

# Set targets
import numpy as np

TARGETS    = ["TEMPERATURE_C", "PRECIPITATION_MM", "WIND_SPEED_KMH"]
INPUT_DAYS = 90



## Pivot to 3D Array & Z-Score Normalize

In [ ]:
# Pivot to (T, n_cities, 3)
cities = sorted(df["CITY"].unique())
frames = [df.pivot(index="DATE", columns="CITY", values=t)[cities].values for t in TARGETS]
data_3d = np.stack(frames, axis=-1)
dates   = df.pivot(index="DATE", columns="CITY", values=TARGETS[0]).index

print(f"data_3d shape: {data_3d.shape}  (T={data_3d.shape[0]}, regions={data_3d.shape[1]}, targets={data_3d.shape[2]})")

# Global Z-score per target
flat = data_3d.reshape(-1, 3)
g_mean = np.nanmean(flat, axis=0)
g_std  = np.nanstd(flat, axis=0)
data_norm = (data_3d - g_mean) / (g_std + 1e-8)

print(f"Mean: {g_mean}")
print(f"Std : {g_std}")

# City one-hot
city_onehot = np.eye(len(cities))

## Build MV Sliding-Window Samples

In [ ]:
# x: (INPUT_DAYS, 9) = 3 targets + 6 city one-hot
# y: (3,)           = next day's 3 targets
T, n_cities, _ = data_norm.shape
samples = []
for city_idx in range(n_cities):
    targets  = data_norm[:, city_idx, :]
    oh_col   = np.tile(city_onehot[city_idx], (T, 1))
    features = np.concatenate([targets, oh_col], axis=1)
    for i in range(T - INPUT_DAYS - 1):
        x = features[i : i + INPUT_DAYS].astype(np.float32)
        y = targets[i + INPUT_DAYS].astype(np.float32)
        samples.append((x, y))

print(f"{len(samples):,} samples  (x: {samples[0][0].shape}, y: {samples[0][1].shape})")


silver_data = {
    "df": df, "data_3d": data_3d, "data_norm": data_norm,
    "mean": g_mean, "std": g_std, "cities": cities,
    "city_onehot": city_onehot, "dates": dates, "samples": samples,
}

## Imports & Best Hyperparameters

In [ ]:
import random
import warnings

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")

# -- Best hyperparameters (MV AR-LSTM) -------------------------------------
INPUT_DAYS         = 90      # lookback window
HIDDEN_SIZE        = 128     # LSTM hidden units
NUM_LAYERS         = 2       # LSTM layers
DROPOUT            = 0.2
BATCH_SIZE         = 32
EPOCHS             = 150
LR                 = 0.001
ES_PATIENCE        = 20      # early stopping patience
TEST_DAYS          = 30      # evaluation holdout
FORECAST_DAYS      = 30      # future forecast horizon
ANOMALY_PERCENTILE = 95

TARGET_LABELS = {
    "TEMPERATURE_C":    "Temperature (C)",
    "PRECIPITATION_MM": "Precipitation (mm)",
    "WIND_SPEED_KMH":   "Wind Speed (km/h)",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Model & Training

In [ ]:
class LSTMAr(nn.Module):
    """1-step-ahead LSTM.  Input (batch, seq, in_dim) -> output (batch, out_dim)."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, HIDDEN_SIZE, NUM_LAYERS, batch_first=True,
                            dropout=DROPOUT if NUM_LAYERS > 1 else 0.0)
        self.fc = nn.Linear(HIDDEN_SIZE, out_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class SampleDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.FloatTensor(x), torch.FloatTensor(y)


def train_model(samples, in_dim, out_dim, label=""):
    """Train one LSTMAr model with early stopping."""
    random.seed(42)
    random.shuffle(samples)
    n_val  = max(1, int(len(samples) * 0.15))
    tr_ld  = DataLoader(SampleDataset(samples[n_val:]),  batch_size=BATCH_SIZE, shuffle=True)
    val_ld = DataLoader(SampleDataset(samples[:n_val]),  batch_size=BATCH_SIZE, shuffle=False)

    model   = LSTMAr(in_dim, out_dim).to(device)
    opt     = torch.optim.Adam(model.parameters(), lr=LR)
    sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=7)
    loss_fn = nn.HuberLoss(delta=1.0)
    best_val, best_state, wait = float("inf"), None, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x_b, y_b in tr_ld:
            x_b, y_b = x_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss_fn(model(x_b), y_b).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        vl = sum(loss_fn(model(x.to(device)), y.to(device)).item()
                 for x, y in val_ld) / len(val_ld)
        if not np.isfinite(vl):
            raise ValueError(
                f"[{label}] validation loss is {vl} at epoch {epoch}. Training "
                "diverged or the input contains NaN — check data_3d for NaN before "
                "training."
            )
        sched.step(vl)

        if epoch % 50 == 0 and label:
            print(f"    [{label}] epoch {epoch}/{EPOCHS}  val={vl:.4f}  lr={opt.param_groups[0]['lr']:.1e}")

        if vl < best_val:
            best_val = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= ES_PATIENCE:
                if label:
                    print(f"    [{label}] early stop at epoch {epoch}  best val={best_val:.4f}")
                break

    if best_state is None:
        raise RuntimeError(
            f"[{label}] no epoch ever improved on the initial validation loss, "
            "so no weights were saved. This normally means the validation loss "
            "was NaN from the first epoch."
        )

    model.load_state_dict(best_state)
    model.to(device).eval()
    return model

## Helper Functions

In [ ]:
def _per_region_zscore(data):
    mean = np.nanmean(data, axis=0)
    std  = np.nanstd(data, axis=0)
    out = (data - mean) / (std + 1e-8)
    if np.isnan(out).any():
        n_bad = int(np.isnan(out).any(axis=(1, 2)).sum())
        raise ValueError(
            f"data_3d contains NaN on {n_bad} of {data.shape[0]} dates. "
            "This means some region is missing rows for those dates, so the "
            "DATE x CITY pivot left holes. Re-run the silver layer, which "
            "drops dates that are not present for every region."
        )
    return out, mean, std


def _mv_samples(data_norm, city_onehot):
    T, nc, _ = data_norm.shape
    samples = []
    for ci in range(nc):
        tgt = data_norm[:, ci, :]
        oh  = np.tile(city_onehot[ci], (T, 1))
        feat = np.concatenate([tgt, oh], axis=1)
        for i in range(T - INPUT_DAYS - 1):
            samples.append((feat[i:i+INPUT_DAYS].astype(np.float32),
                            tgt[i+INPUT_DAYS].astype(np.float32)))
    return samples


def _rollout_mv(model, seed, city_oh, n_steps, noise_std=None, rng_seed=0):
    window = seed.copy()
    preds, rng = [], np.random.default_rng(rng_seed)
    for _ in range(n_steps):
        oh_tiled = np.tile(city_oh, (INPUT_DAYS, 1))
        x = np.concatenate([window, oh_tiled], axis=1).astype(np.float32)
        with torch.no_grad():
            p = model(torch.FloatTensor(x).unsqueeze(0).to(device)).squeeze(0).cpu().numpy()
        if noise_std is not None:
            p = p + rng.normal(0, noise_std).astype(np.float32)
        preds.append(p)
        window = np.vstack([window[1:], p.reshape(1, 3)])
    return np.array(preds)


def _denorm(preds_norm, mean, std):
    raw = preds_norm * (std + 1e-8) + mean
    raw[:, TARGETS.index("PRECIPITATION_MM")] = np.maximum(0.0, raw[:, TARGETS.index("PRECIPITATION_MM")])
    return raw


def _noise_std_mv(model, data_norm, city_onehot, max_samples=2000):
    smp = _mv_samples(data_norm, city_onehot)
    if len(smp) > max_samples:
        idx = np.random.default_rng(0).choice(len(smp), max_samples, replace=False)
        smp = [smp[i] for i in idx]
    xs, ys = np.stack([s[0] for s in smp]), np.stack([s[1] for s in smp])
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(xs), 256):
            preds.append(model(torch.FloatTensor(xs[i:i+256]).to(device)).cpu().numpy())
    return (ys - np.concatenate(preds)).std(axis=0)


## Phase 1: Evaluation — Train on data[:-30], Test Rollout on Last 30 Days

In [ ]:
data_3d     = silver_data["data_3d"]
cities      = silver_data["cities"]
city_onehot = silver_data["city_onehot"]
dates       = silver_data["dates"]
n_cities    = len(cities)
in_dim      = 3 + n_cities

T         = data_3d.shape[0]
train_end = T - TEST_DAYS
train_norm, mean_tr, std_tr = _per_region_zscore(data_3d[:train_end])

print("Training eval model...")
eval_samples = _mv_samples(train_norm, city_onehot)
model_eval   = train_model(eval_samples, in_dim=in_dim, out_dim=3, label="MV eval")

# Test rollout (no noise)
test_actuals = data_3d[train_end:]
mae_per_target = np.zeros(3)
for ci, city in enumerate(cities):
    seed = train_norm[-INPUT_DAYS:, ci, :]
    preds_raw = _denorm(_rollout_mv(model_eval, seed, city_onehot[ci], TEST_DAYS), mean_tr[ci], std_tr[ci])
    for ti, t in enumerate(TARGETS):
        mae_per_target[ti] += mean_absolute_error(test_actuals[:, ci, ti], preds_raw[:, ti])

mae_per_target /= n_cities
print(f"\nTest MAE (avg across {n_cities} regions):")
for ti, t in enumerate(TARGETS):
    print(f"  {TARGET_LABELS[t]:<22} {mae_per_target[ti]:.3f}")


## Phase 2: Full Retrain & 30-Day Forecast with Gaussian Noise

In [ ]:
import pandas as pd

full_norm, mean_full, std_full = _per_region_zscore(data_3d)

print("Retraining on full data...")
model_full = train_model(_mv_samples(full_norm, city_onehot),
                         in_dim=in_dim, out_dim=3, label="MV full")

# Estimate residual noise
ns = _noise_std_mv(model_full, full_norm, city_onehot)
print(f"Noise std (normalised): temp={ns[0]:.3f}  precip={ns[1]:.3f}  wind={ns[2]:.3f}")

# 30-day forecast
last_date = dates[-1]
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq="D")

forecast_rows = []
for ci, city in enumerate(cities):
    seed = full_norm[-INPUT_DAYS:, ci, :]
    preds_raw = _denorm(
        _rollout_mv(model_full, seed, city_onehot[ci], FORECAST_DAYS, noise_std=ns, rng_seed=ci),
        mean_full[ci], std_full[ci])
    for step in range(FORECAST_DAYS):
        forecast_rows.append({
            "DATE": forecast_dates[step], "CITY": city,
            "TEMPERATURE_C": float(preds_raw[step, 0]),
            "PRECIPITATION_MM": float(preds_raw[step, 1]),
            "WIND_SPEED_KMH": float(preds_raw[step, 2]),
            "SOURCE": "forecast",
        })

forecast_df = pd.DataFrame(forecast_rows)
print(f"Forecast: {len(forecast_df)} rows ({forecast_dates[0].date()} -> {forecast_dates[-1].date()})")


## Anomaly Detection & Assemble Gold DataFrame

In [ ]:
# 95th percentile thresholds, computed per region: a national threshold makes
# windy island regions "anomalous" every day and mainland regions never trip,
# so each region is compared against its own historical distribution.
hist_df = silver_data["df"].copy()
th = hist_df.groupby("CITY")[TARGETS].quantile(ANOMALY_PERCENTILE / 100)

print(f"{'Variable':<25} {'Min':>10} {'Median':>10} {'Max':>10}  {'Max region':<30}")
print("-" * 90)
for t in TARGETS:
    col_th = th[t]
    print(f"{TARGET_LABELS[t]:<25} {col_th.min():>10.2f} {col_th.median():>10.2f} "
          f"{col_th.max():>10.2f}  {col_th.idxmax():<30}")

# Flag anomalies (per-region threshold looked up by CITY)
forecast_df["ANOMALY_TEMPERATURE"]   = forecast_df["TEMPERATURE_C"]    > forecast_df["CITY"].map(th["TEMPERATURE_C"])
forecast_df["ANOMALY_PRECIPITATION"] = forecast_df["PRECIPITATION_MM"] > forecast_df["CITY"].map(th["PRECIPITATION_MM"])
forecast_df["ANOMALY_WIND"]          = forecast_df["WIND_SPEED_KMH"]   > forecast_df["CITY"].map(th["WIND_SPEED_KMH"])

# Assemble: historical + forecast
hist_gold = hist_df.copy()
hist_gold["SOURCE"]                = "historical"
hist_gold["ANOMALY_TEMPERATURE"]   = False
hist_gold["ANOMALY_PRECIPITATION"] = False
hist_gold["ANOMALY_WIND"]          = False

gold_df = pd.concat([hist_gold, forecast_df], ignore_index=True)
gold_df = gold_df.sort_values(["CITY", "DATE"]).reset_index(drop=True)

n_hist = (gold_df["SOURCE"] == "historical").sum()
n_fore = (gold_df["SOURCE"] == "forecast").sum()
n_anom = (gold_df["ANOMALY_TEMPERATURE"] | gold_df["ANOMALY_PRECIPITATION"] | gold_df["ANOMALY_WIND"]).sum()
print(f"\ngold_df: {len(gold_df):,} rows ({n_hist:,} historical + {n_fore} forecast, {n_anom} anomalies)")

# -- Use OCI GenAI to generate recommendations ------------------------------
def build_anomaly_summary(row):
    flags = []
    if row["ANOMALY_TEMPERATURE"]:
        flags.append("extreme heat")
    if row["ANOMALY_PRECIPITATION"]:
        flags.append("heavy percipitation")
    if row["ANOMALY_WIND"]:
        flags.append("strong winds")
    return ", ".join(flags) if flags else None

gold_df["combined_anomaly"] = gold_df.apply(build_anomaly_summary, axis=1)



# -- Spark conversion (commented out for future cluster deployment) ----------
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
gold_sdf = spark.createDataFrame(gold_df)


from pyspark.sql.functions import expr, when, col

gold_sdf = gold_sdf.withColumn(
  	"RECOMMENDATION",                                                                                                                                                                          when(
	col("COMBINED_ANOMALY").isNotNull(),
    expr("""query_model(
        'default.oci_ai_models.google.gemini-2.5-pro',
        CONCAT('Give exactly 3 brief safety tips as a numbered list. No introduction, no headers, no markdown formatting. Plain text only. For residents facing:', COMBINED_ANOMALY),
        map('max_tokens', '1024')
    )""")
  	).otherwise(None)
)
gold_sdf.show()

In [ ]:
gold_sdf.filter(col("RECOMMENDATION").isNotNull()).select("RECOMMENDATION").show(1, truncate=False)



In [ ]:
# Create gold catalog
gold_catalog = "gold_weather_india"
gold_schema = "weather"
gold_table = "forecasted_weather"

gold_catalog_adb = "gold_weather_india_adb"
gold_schema_adb = "weather"
gold_table_adb = "FORECASTED_WEATHER"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {gold_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

# Register as temp view first
gold_sdf.createOrReplaceTempView("gold_temp")

# Use SQL DDL to create Delta table
spark.sql(f"""
  CREATE OR REPLACE TABLE {gold_catalog}.{gold_schema}.{gold_table}
  USING DELTA
  AS SELECT * FROM gold_temp
""")

spark.sql(f"""
  CREATE OR REPLACE TABLE {gold_catalog_adb}.{gold_schema_adb}.{gold_table_adb}
  USING DELTA
  AS SELECT * FROM gold_temp
""")

In [ ]:
df = spark.read.table("gold_weather_india.weather.forecasted_weather")
df.show()